# Data Wrangling, I

**Assignment Objective:** Perform data wrangling operations on an open-source dataset using Python.

This notebook demonstrates:
1. Importing required Python libraries
2. Locating an open-source dataset and describing it
3. Loading the dataset into a pandas DataFrame
4. Data preprocessing — checking missing values, initial statistics, variable descriptions, dimensions
5. Data formatting and normalization — checking and converting data types
6. Converting categorical variables into quantitative variables

Data Wrangling I (Titanic Dataset)
Big picture: Take a messy real-world dataset and get it into a clean, fully-numeric shape that any machine learning algorithm can consume.
What we did, step by step:

Brought in our toolbox — Loaded the Python libraries we'd need: one for handling tables, one for math, two for plotting, and one for machine-learning utilities.
Picked a dataset and described it — Chose the famous Titanic passenger dataset from Kaggle. Documented every column: what it represents (passenger ID, ticket class, age, fare, etc.) and what type of variable it is (numeric, categorical, text).
Loaded the data into a table — Read the CSV file from a URL into a structured table (DataFrame) and looked at the first and last few rows to confirm it loaded correctly.
Took inventory of the data:

Checked the dimensions — how many rows and columns we have.
Got a summary of the data — for numeric columns: average, min, max, spread; for text columns: how many unique values, which appears most often.
Hunted for missing values — counted how many cells were empty in each column and worked out the percentage missing.


Filled in the gaps — Different missing values needed different fixes:

Missing ages → filled with the median age (median is safer than average because it isn't dragged around by extreme values).
Missing cabin numbers → most third-class passengers didn't have one recorded, so we filled with the placeholder "Unknown".
Missing port of embarkation → only one or two missing, so we filled with the most common port.


Verified the data types are correct:

Numbers like "Survived" (0/1) and "Pclass" (1/2/3) were technically stored as integers, but they're really categories, not amounts you'd average. We converted them to a proper category type — same idea as a "factor" in R.
Also normalized the numeric columns (age, fare, etc.) to a 0–1 scale so they all play fair when fed to algorithms that care about magnitude.


Turned text categories into numbers — Most ML algorithms can't handle words like "male" or "Cherbourg". We showed three ways to encode them:

Label encoding — assign each category an integer (female=0, male=1).
One-hot encoding — create a separate yes/no column for each category.
Manual mapping — use our own dictionary if we want full control over the order.


Produced a final clean table — Combined everything: no missing values, correct types, all categories converted to numbers, ready for analysis or modeling.

## Step 1: Import all the required Python Libraries

We import the libraries we'll need for data manipulation, numerical computation, visualization, and preprocessing.

In [1]:
# pandas — the primary library for data manipulation and analysis.
# It provides the DataFrame structure (a 2D labeled table, like a spreadsheet)
# and many built-in functions like read_csv(), isnull(), describe(), dtypes, etc.
import pandas as pd

# numpy — fundamental package for numerical computing in Python.
# Provides array objects and mathematical functions; pandas is built on top of numpy.
# We'll use np.nan to represent missing values.
import numpy as np

# matplotlib.pyplot — standard plotting library, used here for any quick visualization.
import matplotlib.pyplot as plt

# seaborn — built on matplotlib, makes statistical plots cleaner and easier.
import seaborn as sns

# sklearn.preprocessing — provides utilities for scaling, normalizing, and encoding data.
# LabelEncoder converts each category in a column to an integer (e.g., male->1, female->0).
# MinMaxScaler scales numeric features to a fixed range, usually [0, 1], which is one form
# of data normalization.
from sklearn.preprocessing import LabelEncoder, MinMaxScaler

# Display settings: show all columns and avoid truncating output (good for inspection).
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

print("All libraries imported successfully.")

All libraries imported successfully.


## Step 2: Dataset Description and Source

### Dataset: Titanic Passenger Data

**Source:** Kaggle — *Titanic: Machine Learning from Disaster*  
URL: https://www.kaggle.com/c/titanic/data

### Description
The Titanic dataset is one of the most popular open-source datasets, used to predict whether a passenger survived the sinking of the RMS Titanic on April 15, 1912. It contains demographic and travel information for 100 passengers in this notebook (a subset of the original 891-row training set).

### Variable Descriptions

| Variable | Description | Type |
|----------|-------------|------|
| **PassengerId** | A unique ID assigned to each passenger | Integer (numeric) |
| **Survived** | Survival status (0 = No, 1 = Yes) — target variable | Integer (categorical, binary) |
| **Pclass** | Ticket class (1 = 1st, 2 = 2nd, 3 = 3rd) — proxy for socio-economic status | Integer (ordinal) |
| **Name** | Passenger's full name | String (text) |
| **Sex** | Passenger's gender (male / female) | String (categorical, nominal) |
| **Age** | Passenger's age in years (fractional if less than 1) | Float (numeric, continuous) |
| **SibSp** | Number of siblings / spouses aboard the Titanic | Integer (numeric, discrete) |
| **Parch** | Number of parents / children aboard the Titanic | Integer (numeric, discrete) |
| **Ticket** | Ticket number | String (text) |
| **Fare** | Passenger fare (in British pounds) | Float (numeric, continuous) |
| **Cabin** | Cabin number | String (categorical, many missing) |
| **Embarked** | Port of embarkation (C = Cherbourg, Q = Queenstown, S = Southampton) | String (categorical, nominal) |

## Step 3: Load the Dataset into a pandas DataFrame

We use `pd.read_csv()` to load the CSV file. This function reads a comma-separated values file and returns a pandas DataFrame.

In [6]:
# URL of the Titanic dataset (same data Kaggle hosts for the competition,
# mirrored on GitHub so it can be loaded over HTTPS without API credentials).
url = 'https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv'

# read_csv() can take a URL directly — pandas will download and parse it.
# A DataFrame is the resulting 2D table with labeled rows (index) and columns.
df = pd.read_csv(url)

print(f"Loaded dataset — shape: {df.shape}")

# head(n) returns the first n rows of the DataFrame (default n=5).
# Useful for a quick look at the data structure and contents.
df.head()

Loaded dataset — shape: (891, 12)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [3]:
# tail(n) returns the last n rows of the DataFrame.
# Useful to confirm the file loaded all the way to the end.
df.tail()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
886,887,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.00,NaN,S
887,888,1,1,"Graham, Miss. Margaret Edith",female,19.0,0,0,112053,30.00,B42,S
888,889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,NaN,1,2,W./C. 6607,23.45,NaN,S
889,890,1,1,"Behr, Mr. Karl Howell",male,26.0,0,0,111369,30.00,C148,C
890,891,0,3,"Dooley, Mr. Patrick",male,32.0,0,0,370376,7.75,NaN,Q


## Step 4: Data Preprocessing

We will:
- Check the dimensions of the DataFrame
- Check data types of each column
- Get summary statistics with `describe()`
- Check for missing values with `isnull()`

### 4.1 Dimensions of the DataFrame

In [7]:
# shape attribute returns a tuple (rows, columns) of the DataFrame.
print("Shape of the DataFrame (rows, columns):", df.shape)

# size attribute returns the total number of elements (rows * columns).
print("Total number of elements:", df.size)

# ndim attribute returns the number of dimensions (always 2 for a DataFrame).
print("Number of dimensions:", df.ndim)

# len(df) gives the number of rows.
print("Number of rows:", len(df))

# df.columns gives the list of column names.
print("Column names:", df.columns.tolist())

Shape of the DataFrame (rows, columns): (891, 12)
Total number of elements: 10692
Number of dimensions: 2
Number of rows: 891
Column names: ['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked']


### 4.2 Information about the DataFrame

In [8]:
# info() prints a concise summary: column names, non-null counts, and data types.
# It also shows the memory usage of the DataFrame.
# This is one of the most useful inspection commands.
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 83.7 KB


### 4.3 Initial Statistics with describe()

In [9]:
# describe() returns summary statistics for all NUMERIC columns by default:
#   count  — number of non-null entries
#   mean   — average
#   std    — standard deviation (spread of the data)
#   min    — minimum value
#   25%    — first quartile (25th percentile)
#   50%    — median (50th percentile)
#   75%    — third quartile (75th percentile)
#   max    — maximum value
df.describe()

,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare
count,891.000000,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,446.000000,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208
std,257.353842,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429
min,1.000000,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,223.500000,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,446.000000,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,668.500000,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,891.000000,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


In [10]:
# describe(include='object') gives statistics for OBJECT (string) columns:
#   count  — number of non-null entries
#   unique — number of distinct values
#   top    — most frequent value
#   freq   — frequency of the most frequent value
df.describe(include='object')

/tmp/ipykernel_10115/3412613460.py:6: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  df.describe(include='object')


,Name,Sex,Ticket,Cabin,Embarked
count,891,891,891,204,889
unique,891,2,681,147,3
top,"Braund, Mr. Owen Harris",male,347082,G6,S
freq,1,577,7,4,644


In [11]:
# describe(include='all') combines numeric and non-numeric statistics in one table.
df.describe(include='all')

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
count,891.000000,891.000000,891.000000,891,891,714.000000,891.000000,891.000000,891,891.000000,204,889
unique,NaN,NaN,NaN,891,2,NaN,NaN,NaN,681,NaN,147,3
top,NaN,NaN,NaN,"Braund, Mr. Owen Harris",male,NaN,NaN,NaN,347082,NaN,G6,S
freq,NaN,NaN,NaN,1,577,NaN,NaN,NaN,7,NaN,4,644
mean,446.000000,0.383838,2.308642,NaN,NaN,29.699118,0.523008,0.381594,NaN,32.204208,NaN,NaN
std,257.353842,0.486592,0.836071,NaN,NaN,14.526497,1.102743,0.806057,NaN,49.693429,NaN,NaN
min,1.000000,0.000000,1.000000,NaN,NaN,0.420000,0.000000,0.000000,NaN,0.000000,NaN,NaN
25%,223.500000,0.000000,2.000000,NaN,NaN,20.125000,0.000000,0.000000,NaN,7.910400,NaN,NaN
50%,446.000000,0.000000,3.000000,NaN,NaN,28.000000,0.000000,0.000000,NaN,14.454200,NaN,NaN
75%,668.500000,1.000000,3.000000,NaN,NaN,38.000000,1.000000,0.000000,NaN,31.000000,NaN,NaN


### 4.4 Checking for Missing Values

In [12]:
# isnull() returns a DataFrame of the same shape with True where values are missing
# (NaN) and False otherwise. It does NOT show the data itself — only the boolean mask.
df.isnull().head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,False,False,False,False,False,False,False,False,False,False,True,False
1,False,False,False,False,False,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False,False,False,True,False
3,False,False,False,False,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False,False,False,True,False


In [10]:
# Combining isnull() with sum() gives the COUNT of missing values per column.
# When you call sum() on booleans, True is treated as 1 and False as 0.
print("Missing values in each column:")
print(df.isnull().sum())

Missing values in each column:
PassengerId     0
Survived        0
Pclass          0
Name            0
Sex             0
Age            22
SibSp           0
Parch           0
Ticket          0
Fare            0
Cabin          80
Embarked        1
dtype: int64


In [13]:
# We can also compute the percentage of missing values per column.
# This helps decide whether to drop a column (too many NaNs) or fill them.
missing_percent = (df.isnull().sum() / len(df)) * 100
print("Percentage of missing values per column:")
print(missing_percent.round(2))

Percentage of missing values per column:
PassengerId     0.00
Survived        0.00
Pclass          0.00
Name            0.00
Sex             0.00
Age            19.87
SibSp           0.00
Parch           0.00
Ticket          0.00
Fare            0.00
Cabin          77.10
Embarked        0.22
dtype: float64


In [15]:
# notnull() is the opposite of isnull() — True where the value is present.
# Useful when filtering rows that have a value in a particular column.
print("Non-null values per column:")
print(df.notnull().sum())

Non-null values per column:
PassengerId    891
Survived       891
Pclass         891
Name           891
Sex            891
Age            714
SibSp          891
Parch          891
Ticket         891
Fare           891
Cabin          204
Embarked       889
dtype: int64


**Observations from missing-value check:**
- `Age` has missing values — we will fill them with the median age (robust to outliers).
- `Cabin` has many missing values (most passengers in 3rd class did not have a recorded cabin) — we will fill with `'Unknown'`.
- `Embarked` has very few missing values — we will fill with the mode (most frequent port).

### 4.5 Handling Missing Values

In [16]:
# fillna() replaces NaN values. We pass the value (or a Series) to substitute.
# inplace=True modifies the DataFrame directly instead of returning a new one.

# Fill missing Age values with the MEDIAN age.
# We use median rather than mean because Age can be skewed by a few very old passengers.
df['Age'].fillna(df['Age'].median(), inplace=True)

# Fill missing Cabin values with the string 'Unknown'.
df['Cabin'].fillna('Unknown', inplace=True)

# mode() returns the most frequent value(s). [0] picks the first one in case of ties.
df['Embarked'].fillna(df['Embarked'].mode()[0], inplace=True)

# Verify there are no missing values left.
print("Missing values after filling:")
print(df.isnull().sum())

Missing values after filling:
PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64


/tmp/ipykernel_10115/3818279366.py:6: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  df['Age'].fillna(df['Age'].median(), inplace=True)
/tmp/ipykernel_10115/3818279366.py:9: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never work

## Step 5: Data Formatting and Data Normalization

Here we check whether each variable has the correct data type, and convert where necessary.

### 5.1 Checking Data Types

In [17]:
# dtypes attribute shows the data type of each column.
# Common pandas dtypes:
#   int64    — integers
#   float64  — floating-point (decimal) numbers
#   object   — strings (text) OR mixed types
#   bool     — True/False
#   category — categorical (factor in R)
#   datetime64 — date/time values
print("Data types of each column:")
print(df.dtypes)

Data types of each column:
PassengerId      int64
Survived         int64
Pclass           int64
Name               str
Sex                str
Age            float64
SibSp            int64
Parch            int64
Ticket             str
Fare           float64
Cabin              str
Embarked           str
dtype: object


In [18]:
# select_dtypes() lets us pick columns by their data type.
# This is useful for separating numeric from non-numeric columns.
print("Numeric columns:")
print(df.select_dtypes(include=['int64', 'float64']).columns.tolist())

print("\nObject (text) columns:")
print(df.select_dtypes(include=['object']).columns.tolist())

Numeric columns:
['PassengerId', 'Survived', 'Pclass', 'Age', 'SibSp', 'Parch', 'Fare']

Object (text) columns:
['Name', 'Sex', 'Ticket', 'Cabin', 'Embarked']


/tmp/ipykernel_10115/3887005708.py:7: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  print(df.select_dtypes(include=['object']).columns.tolist())


### 5.2 Type Conversions

Two columns are stored as integers but represent **categories**, not real numeric quantities:
- `Survived` — 0 / 1 (categorical: did not survive / survived)
- `Pclass` — 1 / 2 / 3 (ordinal categorical: ticket class)

`Sex` and `Embarked` are stored as strings (`object`) but they are categorical.

Converting them to the `category` dtype:
- saves memory
- signals our intent (these are categories, not numbers we'd average)
- works like a *factor* in R

In [19]:
# astype() converts a column from one data type to another.

# Convert Survived (0/1) to category — it's a binary categorical variable, not a count.
df['Survived'] = df['Survived'].astype('category')

# Convert Pclass to category — even though stored as int, classes 1/2/3 are categories.
df['Pclass'] = df['Pclass'].astype('category')

# Convert Sex to category.
df['Sex'] = df['Sex'].astype('category')

# Convert Embarked to category.
df['Embarked'] = df['Embarked'].astype('category')

# Verify the conversions.
print("Data types after conversion:")
print(df.dtypes)

Data types after conversion:
PassengerId       int64
Survived       category
Pclass         category
Name                str
Sex            category
Age             float64
SibSp             int64
Parch             int64
Ticket              str
Fare            float64
Cabin               str
Embarked       category
dtype: object


In [20]:
# For categorical columns, cat.categories shows the unique categories.
print("Categories in 'Sex':", df['Sex'].cat.categories.tolist())
print("Categories in 'Embarked':", df['Embarked'].cat.categories.tolist())
print("Categories in 'Pclass':", df['Pclass'].cat.categories.tolist())
print("Categories in 'Survived':", df['Survived'].cat.categories.tolist())

Categories in 'Sex': ['female', 'male']
Categories in 'Embarked': ['C', 'Q', 'S']
Categories in 'Pclass': [1, 2, 3]
Categories in 'Survived': [0, 1]


### 5.3 Data Normalization (Min-Max Scaling)

In [21]:
# Normalization rescales numeric values to a common range, typically [0, 1].
# Formula: x_scaled = (x - x_min) / (x_max - x_min)
# This is important when features have very different ranges
# (e.g., Age is 0-80 while Fare can go up to 263) so no single feature
# dominates models that rely on distances (like KNN, K-Means).

# Create a copy to keep the original DataFrame intact.
df_normalized = df.copy()

# Pick the continuous numeric columns to normalize.
numeric_cols = ['Age', 'Fare', 'SibSp', 'Parch']

# MinMaxScaler() — scales each column to [0, 1] independently.
scaler = MinMaxScaler()

# fit_transform() learns the min and max from the data and applies the transformation.
df_normalized[numeric_cols] = scaler.fit_transform(df_normalized[numeric_cols])

# Show the normalized values — every numeric column is now in [0, 1].
print("After Min-Max normalization:")
df_normalized[numeric_cols].head()

After Min-Max normalization:


,Age,Fare,SibSp,Parch
0,0.271174,0.014151,0.125,0.0
1,0.472229,0.139136,0.125,0.0
2,0.321438,0.015469,0.000,0.0
3,0.434531,0.103644,0.125,0.0
4,0.434531,0.015713,0.000,0.0


In [22]:
# Confirm the range — min should be 0.0 and max should be 1.0 for every column.
print("After normalization, min and max of each column:")
print(df_normalized[numeric_cols].agg(['min', 'max']))

After normalization, min and max of each column:
     Age  Fare  SibSp  Parch
min  0.0   0.0    0.0    0.0
max  1.0   1.0    1.0    1.0


## Step 6: Turn Categorical Variables into Quantitative Variables

Most machine learning algorithms cannot work with text labels — they need numbers. We have two main techniques:

1. **Label Encoding** — assigns each category an integer (e.g., male=1, female=0). Good for ordinal data or binary variables.
2. **One-Hot Encoding** — creates a new 0/1 column for each category. Good for nominal data with no inherent order, because it doesn't impose a false ordering.

### 6.1 Label Encoding

In [23]:
# Make a copy so we can show two encoding strategies side by side.
df_encoded = df.copy()

# LabelEncoder() assigns an integer to each unique category.
# It alphabetizes the categories and labels them 0, 1, 2, ...
# Example for 'Sex': female -> 0, male -> 1
le_sex = LabelEncoder()
df_encoded['Sex_encoded'] = le_sex.fit_transform(df_encoded['Sex'])

# Show the mapping it learned (which category became which integer).
print("Sex encoding mapping:")
for idx, category in enumerate(le_sex.classes_):
    print(f"  {category} -> {idx}")

# Same idea for Embarked.
le_embarked = LabelEncoder()
df_encoded['Embarked_encoded'] = le_embarked.fit_transform(df_encoded['Embarked'])

print("\nEmbarked encoding mapping:")
for idx, category in enumerate(le_embarked.classes_):
    print(f"  {category} -> {idx}")

# Display the new encoded columns alongside the originals.
df_encoded[['Sex', 'Sex_encoded', 'Embarked', 'Embarked_encoded']].head(10)

Sex encoding mapping:
  female -> 0
  male -> 1

Embarked encoding mapping:
  C -> 0
  Q -> 1
  S -> 2
  nan -> 3


,Sex,Sex_encoded,Embarked,Embarked_encoded
0,male,1,S,2
1,female,0,C,0
2,female,0,S,2
3,female,0,S,2
4,male,1,S,2
5,male,1,Q,1
6,male,1,S,2
7,male,1,S,2
8,female,0,S,2
9,female,0,C,0


### 6.2 One-Hot Encoding (Dummy Variables)

`pd.get_dummies()` is the most common one-hot encoder in pandas. It creates one new column per category, with 1 if the row belongs to that category and 0 otherwise.

In [24]:
# get_dummies() converts categorical variables into multiple 0/1 columns.
# columns=...      — names of the columns to encode
# drop_first=True  — drops the first dummy of each variable to avoid the
#                    "dummy variable trap" (perfect multicollinearity); the dropped
#                    category becomes the reference (baseline).
df_onehot = pd.get_dummies(df, columns=['Sex', 'Embarked', 'Pclass'], drop_first=True)

# Display the new column names.
print("Columns after one-hot encoding:")
print(df_onehot.columns.tolist())

df_onehot.head()

Columns after one-hot encoding:
['PassengerId', 'Survived', 'Name', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Sex_male', 'Embarked_Q', 'Embarked_S', 'Pclass_2', 'Pclass_3']


,PassengerId,Survived,Name,Age,SibSp,Parch,Ticket,Fare,Cabin,Sex_male,Embarked_Q,Embarked_S,Pclass_2,Pclass_3
0,1,0,"Braund, Mr. Owen Harris",22.0,1,0,A/5 21171,7.2500,NaN,True,False,True,False,True
1,2,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",38.0,1,0,PC 17599,71.2833,C85,False,False,False,False,False
2,3,1,"Heikkinen, Miss. Laina",26.0,0,0,STON/O2. 3101282,7.9250,NaN,False,False,True,False,True
3,4,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",35.0,1,0,113803,53.1000,C123,False,False,True,False,False
4,5,0,"Allen, Mr. William Henry",35.0,0,0,373450,8.0500,NaN,True,False,True,False,True


In [25]:
# Without drop_first — every category gets its own column. Use this when you want
# every category visible (e.g., for descriptive analysis rather than modeling).
df_onehot_all = pd.get_dummies(df, columns=['Sex', 'Embarked'])
df_onehot_all[['Sex_female', 'Sex_male', 'Embarked_C', 'Embarked_Q', 'Embarked_S']].head()

,Sex_female,Sex_male,Embarked_C,Embarked_Q,Embarked_S
0,False,True,False,False,True
1,True,False,True,False,False
2,True,False,False,False,True
3,True,False,False,False,True
4,False,True,False,False,True


### 6.3 Manual Mapping (Custom Encoding)

In [26]:
# Sometimes you want full control over the mapping — for example, when the order
# of an ordinal variable matters. map() applies a dictionary of replacements.
df_mapped = df.copy()

# Map Sex manually: female=0, male=1.
df_mapped['Sex_manual'] = df_mapped['Sex'].map({'female': 0, 'male': 1})

# Map Embarked manually: S=0, C=1, Q=2.
df_mapped['Embarked_manual'] = df_mapped['Embarked'].map({'S': 0, 'C': 1, 'Q': 2})

df_mapped[['Sex', 'Sex_manual', 'Embarked', 'Embarked_manual']].head(10)

,Sex,Sex_manual,Embarked,Embarked_manual
0,male,1,S,0
1,female,0,C,1
2,female,0,S,0
3,female,0,S,0
4,male,1,S,0
5,male,1,Q,2
6,male,1,S,0
7,male,1,S,0
8,female,0,S,0
9,female,0,C,1


## Final Cleaned and Encoded DataFrame

We combine all the steps into one final DataFrame ready for analysis or modeling.

In [27]:
# Build the final DataFrame: missing values handled, types corrected, categorical
# variables encoded numerically.
df_final = df.copy()

# Apply label encoding to all categorical columns at once.
for col in ['Sex', 'Embarked', 'Pclass', 'Survived']:
    df_final[col] = LabelEncoder().fit_transform(df_final[col].astype(str))

# Drop columns that aren't useful as numeric features for typical ML
# (high-cardinality text fields).
df_final = df_final.drop(['Name', 'Ticket', 'Cabin'], axis=1)

print("Final DataFrame shape:", df_final.shape)
print("\nFinal data types:")
print(df_final.dtypes)
print("\nFinal DataFrame preview:")
df_final.head(10)

Final DataFrame shape: (891, 9)

Final data types:
PassengerId      int64
Survived         int64
Pclass           int64
Sex              int64
Age            float64
SibSp            int64
Parch            int64
Fare           float64
Embarked         int64
dtype: object

Final DataFrame preview:


,PassengerId,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,1,0,2,1,22.0,1,0,7.2500,2
1,2,1,0,0,38.0,1,0,71.2833,0
2,3,1,2,0,26.0,0,0,7.9250,2
3,4,1,0,0,35.0,1,0,53.1000,2
4,5,0,2,1,35.0,0,0,8.0500,2
5,6,0,2,1,NaN,0,0,8.4583,1
6,7,0,0,1,54.0,0,0,51.8625,2
7,8,0,2,1,2.0,3,1,21.0750,2
8,9,1,2,0,27.0,0,2,11.1333,2
9,10,1,1,0,14.0,1,0,30.0708,0


In [28]:
# Quick statistical check of the fully numeric, fully cleaned DataFrame.
df_final.describe()

,PassengerId,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
count,891.000000,891.000000,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000,891.000000
mean,446.000000,0.383838,1.308642,0.647587,29.699118,0.523008,0.381594,32.204208,1.538721
std,257.353842,0.486592,0.836071,0.477990,14.526497,1.102743,0.806057,49.693429,0.794231
min,1.000000,0.000000,0.000000,0.000000,0.420000,0.000000,0.000000,0.000000,0.000000
25%,223.500000,0.000000,1.000000,0.000000,20.125000,0.000000,0.000000,7.910400,1.000000
50%,446.000000,0.000000,2.000000,1.000000,28.000000,0.000000,0.000000,14.454200,2.000000
75%,668.500000,1.000000,2.000000,1.000000,38.000000,1.000000,0.000000,31.000000,2.000000
max,891.000000,1.000000,2.000000,1.000000,80.000000,8.000000,6.000000,512.329200,3.000000


## Conclusion

In this assignment we have:

1. **Imported** all required libraries (pandas, numpy, matplotlib, seaborn, sklearn).
2. **Located** an open-source dataset (Titanic, from Kaggle) and described every variable.
3. **Loaded** the dataset into a pandas DataFrame using `pd.read_csv()`.
4. **Preprocessed** the data:
   - Checked dimensions with `.shape`
   - Got summary statistics with `.describe()` and `.info()`
   - Detected missing values with `.isnull().sum()`
   - Filled missing values using `median`, `mode`, and a placeholder string
5. **Formatted** the data — checked dtypes with `.dtypes` and converted appropriate columns to `category`. We also normalized numeric features to [0, 1] using `MinMaxScaler`.
6. **Encoded** categorical variables into quantitative ones using three techniques:
   - Label Encoding (`LabelEncoder`)
   - One-Hot Encoding (`pd.get_dummies()`)
   - Manual mapping (`.map()`)

The cleaned and encoded DataFrame is now ready for exploratory data analysis, visualization, or machine learning.
